# 1. Importing the Files

This Python script provides a utility for reading CSV (or gzipped CSV) files from a predefined data directory.


In [ ]:
import os
import pandas as pd
import numpy as np

# CONFIGURATION - change this to your actual data folder
DATA_PATH = "/Users/trivenidhamdhere/Documents/MSc Data Science/Data Science Projects and Methods/Final Course Work/Data/eicu_collaborative_research_database_demo-2.0.1/Final Data"

def load_csv(base_name):
    """
        Load a CSV or gzipped CSV file from the global DATA_PATH directory.

        The function searches for a file named `{base_name}.csv` or `{base_name}.csv.gz`
        in the DATA_PATH directory. It returns the first existing non-empty file as a
        pandas DataFrame. If no matching file is found (or all are empty/missing),
        it returns None.

        Args:
            base_name (str): Base filename without extension (e.g., "patient" or "lab").

        Returns:
            pandas.DataFrame or None: DataFrame containing the CSV data, or None if
            no suitable file exists.

        Notes:
            - The global variable `DATA_PATH` must be defined before calling this function.
            - The function ignores files with size 0 (empty).
            - Uses `low_memory=False` to avoid mixed-type warnings (common in large datasets).
    """
    for ext in ['.csv', '.csv.gz']:
        path = os.path.join(DATA_PATH, base_name + ext)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return pd.read_csv(path, low_memory=False)
    return None


# 2. Loading and Merging Patient, Hospital, and Apache Tables

In [ ]:
# 1. Load main tables
""" 
Purpose:
    - Load the three core tables required for the analysis:
    * patient      - patient-level demographics and outcomes
    * hospital     - hospital metadata (including region)
    * apacheApsVar - physiological variables for acuity adjustment
    - Abort if the patient table is missing (critical dependency).
    - Merge patient data with hospital data on 'hospitalid' to attach
    region information (e.g., hospital region, teaching status) to each patient.

Input (expected files in DATA_PATH):
    - patient.csv (or .csv.gz) - must contain 'hospitalid' and other patient fields.
    - hospital.csv (or .csv.gz) - must contain 'hospitalid' and region columns.
    - apacheApsVar.csv (or .csv.gz) - optional for later modelling.

Output:
    - patient_hosp: pandas DataFrame, patient rows enriched with hospital attributes.
    - apache: raw Apache APS variable table (not merged yet).

Error handling:
   - Raises FileNotFoundError if patient table cannot be loaded.
"""
patient = load_csv('patient')
hospital = load_csv('hospital')
apache = load_csv('apacheApsVar')

if patient is None:
    raise FileNotFoundError("Patient table not found. Check DATA_PATH.")

patient_hosp = patient.merge(hospital, on='hospitalid', how='left')


# 3. FAIRNESS: Demographic Distribution (Cleaned + Robust)

**Purpose:** Assess potential skewness in the training population by summarising ethnicity, gender, and age.

**Input:** `patient` table with columns `ethnicity`, `gender`, `age`

**Processing:**
- For **age**: converts to numeric (coerces `"> 89"`, `"Unknown"`, etc. to `NaN`), drops missing, and reports mean, median, min, max, and count.
- For **categorical** columns (`ethnicity`, `gender`): shows the top 10 categories with counts and percentage of **total patients** (including NaNs), so percentages sum to <100% if missing values exist.

**Output:** Printed summary for each demographic column available.

In [ ]:
#3. FAIRNESS: demographic distribution

print("Fairness - Demographic distribution (training population)")

"""
Purpose:
    - Assess potential bias in the training population by examining the
    distribution of key demographic attributes: ethnicity, gender, and age.
    - Identify under- or over-represented groups that could affect model fairness.

    Input (patient table columns):
    - 'ethnicity' (categorical) - patient's ethnic group.
    - 'gender' (categorical) - patient's gender.
    - 'age' (numeric or string, e.g., "> 89", "Unknown") - patient's age.

Output (printed):
    - For categorical columns:
    * Count and percentage (of total patients, including NaN) for each category.
    * Top 10 categories shown; total number of categories reported if >10.
    - For age:
    * Mean, median, min, max, and number of valid numeric ages.

Robustness & cleaning:
    - Age column is converted to numeric, coercing non-numeric values (e.g., "> 89",
    "Unknown") to NaN before computing summary statistics.
    - Missing data are handled gracefully: if a column is missing or empty,
    a message is printed and the analysis skips that column.
"""

# Define demographic columns and labels
demographics = {
    'ethnicity': 'Ethnicity',
    'gender': 'Gender',
    'age': 'Age (years)'
}

for col, label in demographics.items():
    if col not in patient.columns:
        print(f"No column '{col}' - skipping {label}")
        continue
    
    data = patient[col].dropna()
    if data.empty:
        print(f"Column '{col}' has no valid data - skipping {label}")
        continue
    
    if col == 'age':
        age_numeric = pd.to_numeric(data, errors='coerce')
        age_clean = age_numeric.dropna()
        if age_clean.empty:
            print(f"Column '{col}' had no convertible numeric ages - skipping")
        else:
            print(f"\n{label}: mean={age_clean.mean():.1f}, median={age_clean.median():.1f}, "
                  f"min={age_clean.min()}, max={age_clean.max()}, n={len(age_clean)}")
    else:
        counts = data.value_counts()
        percentages = counts / len(patient) * 100
        print(f"\n{label} (% of total patients):")
        for category, cnt in counts.head(10).items():
            print(f"  {str(category):30s} {cnt:6d} ({percentages[category]:5.1f}%)")
        if len(counts) > 10:
            print(f"  ... and {len(counts)-10} more categories")

FAIRNESS – Demographic distribution (training population)

Ethnicity (% of total patients):
  Caucasian                        2010 ( 79.8%)
  African American                  231 (  9.2%)
  Hispanic                          115 (  4.6%)
  Other/Unknown                      83 (  3.3%)
  Asian                              30 (  1.2%)
  Native American                    12 (  0.5%)

Gender (% of total patients):
  Male                             1508 ( 59.8%)
  Female                           1008 ( 40.0%)

Age (years): mean=62.2, median=65.0, min=15.0, max=89.0, n=2418


# 4. FAIRNESS: Mortality by Region (Context Variation)

**Purpose:** Demonstrate meaningful between-context differences in hospital mortality across US regions.

**Input:** `patient_hosp` (merged table) with columns `region` and `hospitaldischargestatus`

**Processing:**
- Creates `died` flag (1 if `hospitaldischargestatus == 'Expired'` else 0).
- Groups by `region` and computes:
  - `n_patients` = number of patients
  - `n_hospitals` = unique hospitals
  - `mortality_rate` = mean of `died`
- Identifies the region with the smallest patient count and compares its mortality rate to another region.

**Output:** Printed table of region statistics, plus a note highlighting meaningful variation (e.g., small region’s rate differs markedly from a larger one).

In [ ]:
# 4. FAIRNESS: mortality by region
if 'hospitaldischargestatus' in patient_hosp.columns and 'region' in patient_hosp.columns:
    """
        Input:   patient_hosp (merged with hospital, columns: region, hospitaldischargestatus)
        Purpose: Demonstrate meaningful between-context differences.
        Output:  Table of region counts, hospital counts, mortality rates.
    """
    patient_hosp['died'] = (patient_hosp['hospitaldischargestatus'] == 'Expired').astype(float)
    
    region_stats = patient_hosp.groupby('region').agg(
        n_patients=('patientunitstayid', 'count'),
        n_hospitals=('hospitalid', 'nunique'),
        mortality_rate=('died', 'mean')
    ).round(3)
    
    print("Fairness - Mortality rate by US region (context variation)")
    
    print(region_stats.to_string())
    
    min_region = region_stats['n_patients'].idxmin()
    print(f"\nNote: The smallest region ('{min_region}') has only {region_stats.loc[min_region, 'n_patients']} patients, "
          f"but its mortality rate ({region_stats.loc[min_region, 'mortality_rate']*100:.1f}%) "
          f"differs markedly from others (e.g., Midwest: {region_stats.loc['Midwest', 'mortality_rate']*100:.1f}%). "
          f"This validates meaningful between-context variation.")

FAIRNESS – Mortality rate by US region (context variation)
           n_patients  n_hospitals  mortality_rate
region                                            
Midwest           807           62           0.072
Northeast         159           13           0.119
South             738           54           0.085
West              606           39           0.083

Note: The smallest region ('Northeast') has only 159 patients, but its mortality rate (11.9%) differs markedly from others (e.g., Midwest: 7.2%). This validates meaningful between‑context variation.


# 5. PRIVACY: Small Subgroups (Hospitals with <10 Patients)

**Purpose:** Assess re-identification risk by identifying hospitals with very few patients.

**Input:** `patient` table with `hospitalid` column

**Processing:**
- Counts patients per hospital using `value_counts()`.
- Flags hospitals with fewer than 10 patients.
- Reports total hospitals, percentage of tiny hospitals, and lists the smallest ones.

**Output:** Summary statistics and a warning that even de-identified data from small units may be traceable.

In [ ]:
# 5. PRIVACY: small subgroups
""" 
    Purpose:
        - Assess re-identification risk by identifying hospitals with very few
        patients (<10). Small group sizes make it easier to link de-identified
        records to real individuals using external data.

    Input (patient table):
        - 'hospitalid' - identifier for each hospital.

    Output (printed):
        - Total number of hospitals.
        - Number and percentage of hospitals with <10 patients.
        - List of the 10 smallest hospitals (by patient count) if any exist.
        - Brief interpretation of the risk.

    Privacy implication:
        - Even de-identified data from tiny hospitals may be traceable.
        For example, a hospital with 3 lung-cancer patients of a certain age
        could be uniquely identifiable in public records.
"""
print("Privacy - Hospital size distribution (re-identification risk)")


if 'hospitalid' in patient.columns:
    hosp_counts = patient['hospitalid'].value_counts()
    tiny = hosp_counts[hosp_counts < 10]
    print(f"Total hospitals: {len(hosp_counts)}")
    print(f"Hospitals with <10 patients: {len(tiny)} ({len(tiny)/len(hosp_counts)*100:.1f}%)")
    if len(tiny) > 0:
        print("\nSmallest hospitals (patients):")
        print(tiny.sort_values().head(10).to_string())
        print("\n Re-identification risk: even de-identified data from such small units may be traceable.")
    else:
        print("No hospital with <10 patients in this demo - but full database may have them.")


PRIVACY - Hospital size distribution (re-identification risk)
Total hospitals: 186
Hospitals with <10 patients: 0 (0.0%)
No hospital with <10 patients in this demo - but full database may have them.


# 6. GENERAL LIMITATIONS: US-Only and Limited Diversity

**Purpose:** Explicitly state that the data comes solely from US hospitals and has limited ethnic diversity, affecting generalizability.

**Input:** `patient_hosp` (with `region` column) and `patient` (with `ethnicity` column)

**Processing:**
- Lists unique US regions observed.
- Computes the proportion of patients with ethnicity containing “Caucasian” (case-insensitive).

**Output:**  
- Statement that all hospitals are in the US, with observed regions listed.  
- Warning that results may not transfer to non-US systems.  
- Percentage of Caucasian patients, highlighting limited diversity for fairness testing.

In [ ]:
# 6. GENERAL LIMITATIONS: US-only and limited diversity

""" 
    Purpose:
        - Explicitly document two major limitations of the dataset:
        1. All hospitals are located in the US, limiting generalisability
        to non‑US healthcare systems.
        2. The patient population is predominantly Caucasian, which limits
        the ability to test for fairness across diverse ethnic groups.

    Input:
        - patient_hosp DataFrame with 'region' column (US regions)
        - patient DataFrame with 'ethnicity' column

    Output (printed):
        - List of observed US regions in the data.
        - Statement that results may not transfer internationally.
        - Percentage of patients with ethnicity containing "Caucasian".
        - Warning about limited diversity for fairness testing.

    Rationale:
        - Recognising these limitations is essential before any modelling or
    inference that aims to generalise beyond the source population.
"""

print("General limitations")


if 'region' in patient_hosp.columns:
    regions = patient_hosp['region'].dropna().unique()
    print(f"All hospitals are in the US (regions observed: {sorted(regions)}).")
    print("Results may not transfer to non-US healthcare systems.")

if 'ethnicity' in patient.columns:
    caucasian_pct = (patient['ethnicity'].str.contains('Caucasian', case=False, na=False).sum() / len(patient)) * 100
    print(f"\nCaucasian proportion: {caucasian_pct:.1f}% - limited diversity for fairness testing.")

GENERAL LIMITATIONS
All hospitals are in the US (regions observed: ['Midwest', 'Northeast', 'South', 'West']).
Results may not transfer to non-US healthcare systems.

Caucasian proportion: 79.8% - limited diversity for fairness testing.


# 7. Additional Check: Outcome Prevalence (Prior Probability Shift)

**Purpose:** Detect potential prior probability shift by calculating overall in-hospital mortality.

**Input:** `patient` table with `hospitaldischargestatus` column

**Processing:**  
- Computes proportion of patients with `hospitaldischargestatus == 'Expired'`.

**Output:**  
- Prints mortality percentage.  
- Warns that if a deployment setting has a very different mortality rate, prior probability shift may occur, affecting model calibration and performance.

In [ ]:
# 7. Additional check: outcome prevalence 
"""
Purpose:
    - Estimate the overall in‑hospital mortality rate in the dataset.
    - Warn about potential prior probability shift if the deployment
        population has a substantially different mortality rate.

Input (patient table):
    - 'hospitaldischargestatus' – categorical variable indicating patient
    discharge disposition (e.g., 'Expired' vs. 'Alive').

Output (printed):
    - Overall in‑hospital mortality percentage.
    - Note that if the target population's mortality differs greatly,
    model calibration and decision thresholds may need adjustment.

Relevance for deployment:
    - Prior probability shift (a type of dataset shift) occurs when the
    base rate of the outcome changes between training and deployment.
    - This can degrade probabilistic predictions and threshold‑based
    decisions (e.g., alerting on high predicted risk). 
"""

if 'hospitaldischargestatus' in patient.columns:
    mortality = (patient['hospitaldischargestatus'] == 'Expired').mean()
    print(f"\nOverall in-hospital mortality: {mortality*100:.1f}%")


Overall in‑hospital mortality: 8.4%
If deployment setting has very different mortality, prior probability shift may occur.
